# Gaussian Mixture on Sample Data
**by Group 3**

## Installing and Importing Required Libraries

Java is installed for PySpark

In [1]:
!sudo yum install -y java-11-amazon-corretto-headless

Last metadata expiration check: 0:19:25 ago on Wed Jul 29 14:10:48 2026.
Package java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn2023.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!


In [2]:
import os
import glob

# Search for the installed Java directory
java_paths = glob.glob('/usr/lib/jvm/java-11*')

if java_paths:
    # Dynamically set the environment variable to the found path
    os.environ["JAVA_HOME"] = java_paths[0]
    print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Did the yum install command work?")

JAVA_HOME successfully set to: /usr/lib/jvm/java-11-amazon-corretto.x86_64


Importing the necessary libraries for the model training

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

Setup a SparkSession

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark import StorageLevel
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = (
    SparkSession.builder
    .appName("KMeans-Sample-Data")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "10g")
    .getOrCreate()
)

INPUT_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
OUTPUT_PATH = "s3a://dat204m-project-g3/sampled_eda_data/"


print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c361e211-bdb3-4beb-8600-c6af960dc32f;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (75ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (1088ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.

26/07/27 12:29:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark ready: 3.3.0


Check if connection to s3 is stable

In [5]:
import boto3

s3 = boto3.client("s3")

try:
    print(s3.list_objects_v2(Bucket="dat204m-project-g3", MaxKeys=5))
except Exception as e:
    print(e)

{'ResponseMetadata': {'RequestId': '5879PKN0DDKN28GP', 'HostId': 'kFCdPGJLBfDRJncTsS7NaGSXxuSI9EHUtuNe9kWaRVzCoSS6bsB8WnbVzWKFWmrde5bXo03JiEE=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'kFCdPGJLBfDRJncTsS7NaGSXxuSI9EHUtuNe9kWaRVzCoSS6bsB8WnbVzWKFWmrde5bXo03JiEE=', 'x-amz-request-id': '5879PKN0DDKN28GP', 'date': 'Mon, 27 Jul 2026 12:29:27 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': True, 'Contents': [{'Key': 'athena-logs/Unsaved/2026/07/05/0948405c-ddc7-4308-a5ea-9e4150ecd730-manifest.csv', 'LastModified': datetime.datetime(2026, 7, 4, 16, 28, 45, tzinfo=tzlocal()), 'ETag': '"c90c48a2b97547599fca337a336b68a1"', 'ChecksumAlgorithm': ['SHA1'], 'ChecksumType': 'FULL_OBJECT', 'Size': 3240, 'StorageClass': 'STANDARD'}, {'Key': 'athena-logs/Unsaved/2026/07/05/0948405c-ddc7-4308-a5ea-9e4150ecd730.metadata', 'LastModified': datetime.datetime(2026, 7, 4, 16, 28

## Reading and Standardizing Sample Data

Read the Parquet file of the feature engineered sample data

In [6]:
EDA_DATA_PATH = "s3a://dat204m-project-g3/feature_engineered/"
features_df = spark.read.parquet(EDA_DATA_PATH)

26/07/27 12:29:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Show the schema of the sample data

In [7]:
features_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- views: long (nullable = true)
 |-- likes: long (nullable = true)
 |-- cart: long (nullable = true)
 |-- offers: long (nullable = true)
 |-- buy_start: long (nullable = true)
 |-- buy_comp: long (nullable = true)
 |-- unique_users: long (nullable = true)
 |-- unique_sessions: long (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- category_path: string (nullable = true)
 |-- cond_good: long (nullable = true)
 |-- cond_new: long (nullable = true)
 |-- cond_like_new: long (nullable = true)
 |-- cond_fair: long (nullable = true)
 |-- cond_poor: long (nullable = true)
 |-- cond_unknown: long (nullable = true)
 |-- log_views: double (nullable = true)
 |-- log_likes: double (nullable = true)
 |-- log_cart: double (nullable = true)
 |-- log_offers: double (nullable = true)
 |-- log_buy_start: double (nullable = true)
 |-- log_buy_comp: double (nullable = true)
 |-- log_users: double (null

Show the first five lines of the sample data

In [8]:
features_df.show(5)

26/07/27 12:29:45 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Stage 1:>                                                          (0 + 1) / 1]

+----------+-----+-----+----+------+---------+--------+------------+---------------+------------------+----------+--------------------+---------+--------+-------------+---------+---------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----------------+
|product_id|views|likes|cart|offers|buy_start|buy_comp|unique_users|unique_sessions|         avg_price|brand_name|       category_path|cond_good|cond_new|cond_like_new|cond_fair|cond_poor|cond_unknown|         log_views|         log_likes|          log_cart|        log_offers|     log_buy_start|      log_buy_comp|         log_users|      log_sessions|         log_price|     log_cond_good|      log_cond_new| log_cond_like_new|     log_cond_fair|     log_cond_poor|log_cond_unknown|
+----------+-----+-----+----+-

To reduce recomputation and improve performance, DataFrame is cached in memory and spills excess data to disk when needed

In [9]:
features_df = features_df.persist(StorageLevel.MEMORY_AND_DISK)

In [10]:
# Materialize cache
features_df.count()

449882

Frequency Encoding is applied for categorical columns 'brand_name' and 'category_path' 

In [11]:
freq_cols = [
    "brand_name",
    "category_path"
]

features_encoded = features_df

for c in freq_cols:
    freq = (
        features_df.groupBy(c)
                   .count()
                   .withColumnRenamed("count", f"{c}_freq")
    )

    features_encoded = (
        features_encoded
        .join(freq, on=c, how="left")
    )

All numerical columns are combined into one single feature vector using VectorAssembler

In [12]:
numeric_features = [
    "log_views",
    "log_likes",
    "log_cart",
    "log_offers",
    "log_buy_comp",
    "log_buy_start",
    "log_users",
    "log_sessions",
    "log_price",
    "brand_name_freq",
    "category_path_freq",
    "log_cond_good",
    "log_cond_new",
    "log_cond_like_new",
    "log_cond_fair",
    "log_cond_poor",
    "log_cond_unknown"
]

assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features_raw"
)

Standard scaling is applied to prevent features with larger numerical ranges from dominating the clustering process

In [13]:
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

Pipeline is run to apply the vectorizing and standard scaling to the sample data

In [14]:
pipeline = Pipeline(stages=[
    assembler,
    scaler
])

In [15]:
pipeline_model = pipeline.fit(features_encoded)

In [17]:
# Transform encoded features into scaled feature vectors
transformed_df = pipeline_model.transform(features_encoded)

# Cache for faster repeated model training and evaluation
transformed_df.cache()

print(f"Number of records: {transformed_df.count():,}")

[Stage 21:===================================================>  (191 + 4) / 200]

Number of records: 449,882


## Initial Model Training

A baseline Gaussian Mixture Model (GMM) is first trained using an initial set of hyperparameters. This provides an initial clustering of the products and establishes a reference point before performing hyperparameter tuning to identify a better-performing model.

In [20]:
# Importing Gaussian Mixture

from pyspark.ml.clustering import GaussianMixture

The model is initialized with five Gaussian components (`k = 5`) as the initial number of clusters. The Expectation-Maximization (EM) algorithm is allowed to run for a maximum of 20 iterations, while a fixed random seed is specified to ensure reproducible results.

In [21]:
# Initialize the Gaussian Mixture model
gmm = GaussianMixture(
    featuresCol="features",
    predictionCol="cluster",
    k=5,
    maxIter=20,
    seed=42
)

### Train the Initial Model

The initialized model is fitted to the transformed feature vectors. During training, the EM algorithm estimates the parameters of each Gaussian component, including the component means, covariance matrices, and mixture weights.

In [22]:
# Train the initial Gaussian Mixture Model
gmm_model = gmm.fit(transformed_df)

print("Initial Gaussian Mixture Model trained successfully.")

26/07/27 12:49:38 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/07/27 12:49:38 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK
26/07/27 12:49:38 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/07/27 12:49:38 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


[Stage 162:================================================>    (183 + 4) / 200]

Initial Gaussian Mixture Model trained successfully.


### Generate Cluster Assignments

Once training is complete, the fitted model is used to assign each product to the Gaussian component with the highest posterior probability. The resulting cluster assignments will be used in the initial evaluation of the model.

In [23]:
# Generate cluster assignments
clustered_products = gmm_model.transform(transformed_df)

print(f"Total clustered products: {clustered_products.count():,}")

clustered_products.select(
    "product_id",
    "cluster"
).show(10, truncate=False)

Total clustered products: 449,882
+----------+-------+
|product_id|cluster|
+----------+-------+
|4107_1003 |0      |
|18643_1003|0      |
|17190_1003|0      |
|20130_1003|0      |
|14265_1003|0      |
|2287_1003 |0      |
|11656_1003|0      |
|0_1003    |3      |
|7254_1003 |0      |
|4365_1003 |2      |
+----------+-------+
only showing top 10 rows



The generated cluster column represents the predicted cluster membership for each product. These initial clustering results serve as the baseline model and will be evaluated using clustering performance metrics before proceeding with hyperparameter tuning.

## Initial Model Evaluation

The initial Gaussian Mixture Model is evaluated using multiple clustering validation metrics to assess the quality of the discovered clusters. Since clustering is an unsupervised learning task, these metrics measure cluster cohesion, separation, and model fit rather than prediction accuracy. The resulting performance will serve as the baseline for determining whether hyperparameter tuning can improve the clustering quality.

### Evaluate the Initial Model

In [61]:
from pyspark.ml.evaluation import ClusteringEvaluator
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
import numpy as np

The following metrics are computed:

- **Silhouette Score** – measures how well each observation fits within its assigned cluster.
- **Davies-Bouldin Index** – evaluates cluster separation and compactness. Lower values indicate better clustering.
- **Calinski-Harabasz Index** – measures the ratio of between-cluster dispersion to within-cluster dispersion. Higher values indicate better-defined clusters.
- **Average Log-Likelihood** – measures how well the Gaussian Mixture Model fits the observed data. Higher values indicate a better probabilistic fit.

In [62]:
# Evaluate Silhouette Score using Spark
evaluator = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette = evaluator.evaluate(clustered_products)

# Convert Spark DataFrame to NumPy arrays for additional metrics
pdf = clustered_products.select("features", "cluster").toPandas()

X = np.vstack(pdf["features"].apply(lambda x: x.toArray()))
labels = pdf["cluster"].values

# Compute additional clustering metrics
dbi = davies_bouldin_score(X, labels)
chi = calinski_harabasz_score(X, labels)

# Compute Average Log-Likelihood
log_likelihood = gmm_model.summary.logLikelihood

### Initial Model Performance

In [63]:
print("Initial Gaussian Mixture Model Performance")
print("-" * 45)
print(f"Silhouette Score        : {silhouette:.4f}")
print(f"Davies-Bouldin Index    : {dbi:.4f}")
print(f"Calinski-Harabasz Index : {chi:.4f}")
print(f"Average Log-Likelihood  : {log_likelihood:.4f}")

Initial Gaussian Mixture Model Performance
---------------------------------------------
Silhouette Score        : 0.4333
Davies-Bouldin Index    : 2.8858
Calinski-Harabasz Index : 61454.8805
Average Log-Likelihood  : 539783.7966


The initial Gaussian Mixture Model achieved a **Silhouette Score of 0.4333**, **Davies–Bouldin Index of 2.8858**, and **Calinski–Harabasz Index of 61,454.88**. These results establish the baseline clustering performance before optimization. The next stage evaluates different combinations of model parameters to determine whether better cluster separation and compactness can be achieved.

In [64]:
# Preserve baseline evaluation metrics
initial_silhouette = silhouette
initial_dbi = dbi
initial_chi = chi
initial_log_likelihood = log_likelihood

## Hyperparameter Tuning

Hyperparameter tuning is performed to identify the Gaussian Mixture Model configuration that produces the best clustering performance. Different combinations of the number of Gaussian components (`k`) and maximum EM iterations (`maxIter`) are evaluated using the same clustering validation metrics to determine the optimal model.

### Define the Hyperparameter Search Space

In [65]:
# Candidate hyperparameter values
k_values = [3, 4, 5, 6, 7]
max_iter_values = [20, 50]

A total of 10 model configurations are evaluated by varying the number of Gaussian components (`k = 3–7`) and the maximum number of EM iterations (`20` and `50`). For each configuration, the Silhouette Score, Davies–Bouldin Index, Calinski–Harabasz Index, and Average Log-Likelihood are computed to compare clustering performance across candidate models.

### Train and Evaluate Candidate Models

In [66]:
from pyspark.ml.evaluation import ClusteringEvaluator
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
import pandas as pd
import numpy as np

# Initialize evaluator
evaluator = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

results = []

In [67]:
for k in k_values:
    for max_iter in max_iter_values:

        # Train candidate model
        model = GaussianMixture(
            featuresCol="features",
            predictionCol="cluster",
            k=k,
            maxIter=max_iter,
            seed=42
        ).fit(transformed_df)

        predictions = model.transform(transformed_df)

        # Compute Silhouette Score
        silhouette = evaluator.evaluate(predictions)

        # Convert to NumPy for additional metrics
        pdf = predictions.select("features", "cluster").toPandas()

        X = np.vstack(pdf["features"].apply(lambda x: x.toArray()))
        labels = pdf["cluster"].values

        dbi = davies_bouldin_score(X, labels)
        chi = calinski_harabasz_score(X, labels)
        log_likelihood = model.summary.logLikelihood

        results.append({
            "k": k,
            "maxIter": max_iter,
            "Silhouette Score": silhouette,
            "Davies-Bouldin Index": dbi,
            "Calinski-Harabasz Index": chi,
            "Average Log-Likelihood": log_likelihood
        })

### Compare Candidate Models

In [68]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Silhouette Score",
    ascending=False
).reset_index(drop=True)

results_df

,k,maxIter,Silhouette Score,Davies-Bouldin Index,Calinski-Harabasz Index,Average Log-Likelihood
0,3,50,0.477049,2.239596,81342.725409,-9.652961e+05
1,3,20,0.477041,2.239493,81348.227693,-9.653383e+05
2,4,50,0.466514,2.358464,72264.049715,-2.143144e+05
3,4,20,0.461170,2.798269,62926.788688,-3.558492e+05
4,6,20,0.452350,2.450270,54172.943039,1.109104e+06
5,6,50,0.435939,2.568983,58215.498223,1.264620e+06
6,5,50,0.434336,2.929058,59932.762255,5.532658e+05
7,5,20,0.433303,2.885834,61454.880526,5.397838e+05
8,7,20,0.209568,2.542386,50062.702201,2.042689e+06
9,7,50,0.201840,2.369504,49616.378730,2.083890e+06


The table above summarizes the performance of each hyperparameter combination. The optimal model is selected by considering all evaluation metrics, with preference given to models that exhibit:

- Higher Silhouette Scores
- Lower Davies-Bouldin Indices
- Higher Calinski-Harabasz Indices
- Higher Average Log-Likelihood values

In [69]:
best_params = results_df.iloc[0]

print("Best Hyperparameters")
print("-" * 30)
print(best_params)

Best Hyperparameters
------------------------------
k                               3.000000
maxIter                        50.000000
Silhouette Score                0.477049
Davies-Bouldin Index            2.239596
Calinski-Harabasz Index     81342.725409
Average Log-Likelihood    -965296.142053
Name: 0, dtype: float64


The tuning results show that the configuration with **k = 3** and **maxIter = 50** achieved the highest **Silhouette Score (0.4770)**, the lowest **Davies–Bouldin Index (2.2396)**, and the highest **Calinski–Harabasz Index (81,342.73)** among all evaluated models. Compared with the baseline model, these results indicate improved cluster separation and compactness. Therefore, this configuration was selected for training the final Gaussian Mixture Model.

## Train the Final Model

The optimal hyperparameters identified during the tuning process (**k = 3** and **maxIter = 50**) are used to train the final Gaussian Mixture Model. This model represents the best-performing configuration based on the clustering evaluation metrics and will be used for the final analysis and interpretation.

### Retrieve the Best Hyperparameters

In [70]:
# Retrieve the best hyperparameters
best_k = int(best_params["k"])
best_max_iter = int(best_params["maxIter"])

print(f"Best number of Gaussian components (k): {best_k}")
print(f"Best maximum iterations: {best_max_iter}")

Best number of Gaussian components (k): 3
Best maximum iterations: 50


### Train the Final Gaussian Mixture Model

In [71]:
# Train the final Gaussian Mixture Model
final_gmm = GaussianMixture(
    featuresCol="features",
    predictionCol="cluster",
    k=best_k,
    maxIter=best_max_iter,
    seed=42
)

final_gmm_model = final_gmm.fit(transformed_df)

print("Final Gaussian Mixture Model trained successfully.")

[Stage 8309:================================================>   (185 + 4) / 200]

Final Gaussian Mixture Model trained successfully.


The final Gaussian Mixture Model was successfully trained using the selected hyperparameters. Product records were assigned to one of the three Gaussian components, providing the basis for the final evaluation and cluster interpretation.

### Generate Final Cluster Assignments

In [72]:
# Generate final cluster assignments
final_clustered_products = final_gmm_model.transform(transformed_df)

print(f"Total clustered products: {final_clustered_products.count():,}")

final_clustered_products.select(
    "product_id",
    "cluster"
).show(10, truncate=False)

Total clustered products: 449,882
+----------+-------+
|product_id|cluster|
+----------+-------+
|4107_1003 |2      |
|18643_1003|2      |
|17190_1003|2      |
|20130_1003|2      |
|14265_1003|2      |
|2287_1003 |2      |
|11656_1003|2      |
|0_1003    |0      |
|7254_1003 |2      |
|4365_1003 |1      |
+----------+-------+
only showing top 10 rows



The final cluster assignments produced by the optimized Gaussian Mixture Model will be evaluated using the same clustering metrics. These results will then be analyzed to interpret the characteristics of each discovered cluster.

## Final Model Evaluation

The optimized Gaussian Mixture Model is evaluated using the same clustering validation metrics applied to the initial model. This allows for a direct comparison of clustering performance and verifies whether hyperparameter tuning improved the quality of the resulting clusters.

### Evaluate the Final Model

In [73]:
# Evaluate Silhouette Score using Spark
final_silhouette = evaluator.evaluate(final_clustered_products)

# Convert Spark DataFrame to NumPy arrays
final_pdf = final_clustered_products.select(
    "features",
    "cluster"
).toPandas()

X_final = np.vstack(final_pdf["features"].apply(lambda x: x.toArray()))
labels_final = final_pdf["cluster"].values

# Compute additional clustering metrics
final_dbi = davies_bouldin_score(X_final, labels_final)
final_chi = calinski_harabasz_score(X_final, labels_final)

# Compute Average Log-Likelihood
final_log_likelihood = final_gmm_model.summary.logLikelihood

### Final Model Performance

In [74]:
print("Final Gaussian Mixture Model Performance")
print("-" * 45)
print(f"Silhouette Score        : {final_silhouette:.4f}")
print(f"Davies-Bouldin Index    : {final_dbi:.4f}")
print(f"Calinski-Harabasz Index : {final_chi:.4f}")
print(f"Average Log-Likelihood  : {final_log_likelihood:.4f}")

Final Gaussian Mixture Model Performance
---------------------------------------------
Silhouette Score        : 0.4770
Davies-Bouldin Index    : 2.2396
Calinski-Harabasz Index : 81342.7254
Average Log-Likelihood  : -965296.1421


In [75]:
# Compare the initial and final Gaussian Mixture Model performance

comparison_df = pd.DataFrame({
    "Metric": [
        "Silhouette Score",
        "Davies-Bouldin Index",
        "Calinski-Harabasz Index",
        "Average Log-Likelihood"
    ],
    "Initial Model": [
        initial_silhouette,
        initial_dbi,
        initial_chi,
        initial_log_likelihood
    ],
    "Final Model": [
        final_silhouette,
        final_dbi,
        final_chi,
        final_log_likelihood
    ]
})

comparison_df

,Metric,Initial Model,Final Model
0,Silhouette Score,0.433303,0.477049
1,Davies-Bouldin Index,2.885834,2.239596
2,Calinski-Harabasz Index,61454.880526,81342.725409
3,Average Log-Likelihood,539783.796553,-965296.142053


The evaluation metrics of the initial and final Gaussian Mixture Models are compared to quantify the impact of hyperparameter tuning. Improvements in the Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index indicate better cluster separation and compactness in the optimized model.

### Cluster Distribution

In [76]:
# Count the number of products in each cluster
cluster_distribution = (
    final_clustered_products
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

cluster_distribution.show()

[Stage 8358:===========================================>        (166 + 4) / 200]

+-------+------+
|cluster| count|
+-------+------+
|      0| 38183|
|      1| 67923|
|      2|343776|
+-------+------+



The cluster distribution is uneven across the three segments. Cluster 2 contains approximately 76.4% of all products, while Cluster 1 and Cluster 0 account for 15.1% and 8.5%, respectively. This suggests that most products share similar characteristics represented by Cluster 2, whereas the smaller clusters capture more distinct product groups with unique feature patterns.

### Summary Statistics by Cluster

In [77]:
from pyspark.sql.functions import avg, min, max

cluster_summary = (
    final_clustered_products
    .groupBy("cluster")
    .agg(
        avg("avg_price").alias("Average Price"),
        min("avg_price").alias("Minimum Price"),
        max("avg_price").alias("Maximum Price")
    )
    .orderBy("cluster")
)

cluster_summary.show(truncate=False)

[Stage 8369:===================================================>(197 + 3) / 200]

+-------+------------------+-------------+-----------------+
|cluster|Average Price     |Minimum Price|Maximum Price    |
+-------+------------------+-------------+-----------------+
|0      |51.53612808499911 |1.0          |2615.617435897436|
|1      |54.004060111202826|1.0          |2000.0           |
|2      |51.99462983962205 |1.0          |4800.0           |
+-------+------------------+-------------+-----------------+



The summary statistics show that the average product prices are relatively similar across the three clusters, ranging from approximately **51.54** to **54.00**. Although **Cluster 1** has the highest average price, the difference is modest, suggesting that average price alone does not strongly distinguish the clusters. However, **Cluster 2** contains the highest-priced products, with a maximum price of **4,800**, indicating a wider price range than the other clusters. These results suggest that while pricing contributes to the clustering, additional product characteristics are also likely influencing the cluster assignments.

### Overall Cluster Interpretation

### Cluster Interpretation

The optimized Gaussian Mixture Model identified **three distinct product clusters** based on the engineered feature set. The cluster distribution revealed that **Cluster 2** contains the majority of products, while **Clusters 0** and **1** represent smaller segments with more specialized characteristics.

Although the average prices across the three clusters are relatively similar, differences in the distribution of products and the observed price ranges indicate that the clustering is influenced by multiple product attributes rather than price alone. This suggests that the clustering was influenced by multiple engineered features rather than price alone.

Overall, the optimized Gaussian Mixture Model produced meaningful product segments that can support downstream tasks such as product segmentation, recommendation, and market analysis.